In [1]:
import numpy as np
import cv2

In [2]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [3]:
# READ IMAGE WITH ALPHA CHANNEL'
# cv2.imread_unchanged -keeps 4 channels (BGR and alpha)
# sunglasses=cv2.imread("sunglasses.jpeg", cv2.IMREAD_UNCHANGED)
sunglasses = cv2.imread("sunglass.png", cv2.IMREAD_UNCHANGED)
print(sunglasses.shape)

(360, 560)


In [4]:
cap = cv2.VideoCapture(0)
while True:
    flag, frame = cap.read()

    if not flag:
        break
        
    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray,1.3)

    for x,y,w,h in faces:
        # Resize sunglasses according to face
        overlay_width = w             # width = face width
        overlay_height = int(h*0.4)   # height = ~40% of face height

        resized_sunglasses = cv2.resize(sunglasses,(overlay_width,overlay_height))
        y_offset = y+h//4 # roughly eye region
        x_offset = x

        #Handle Transparency(Alpha Blending)
        # Split jpg into color (BGR)
        overlay_img = cv2.cvtColor(resized_sunglasses,cv2.COLOR_GRAY2BGR)

        # Create mask from grayscale image
        gray_glass = resized_sunglasses
        _, mask = cv2.threshold(gray_glass,240,255,cv2.THRESH_BINARY_INV)

        # Create inverse mask
        mask_inv = cv2.bitwise_not(mask)

        #Define ROI- Region Of Interest on frame
        # This is where we will place the sunglasses
        roi = frame[y_offset:y_offset+overlay_height,x_offset:x_offset+overlay_width]

        # Extract background from ROI
        bg = cv2.bitwise_and(roi,roi,mask = mask_inv)

        #Extract foreground
        fg = cv2.bitwise_and(overlay_img,overlay_img,mask= mask)

        combined = cv2.add(bg,fg)
        frame[y_offset: y_offset + overlay_height, x_offset: x_offset + overlay_width] = combined

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,225,0),2)

    cv2.imshow("sunglasses",frame)
    key = cv2.waitKey(1) & 0xFF
    if key==27:
        break

cap.release()
cv2.destroyAllWindows()